In [ ]:
# Sandhi — Generative Waveform Reconstruction for Lossy Audio

**Architecture:** 1-D Convolutional U-Net Generator + 1-D Conv Discriminator (GAN)  
**Dataset:** LJ-Speech 1.1 (22 050 Hz, mono, peak-normalised)  
**Task:** Given a raw waveform with a zeroed-out gap (simulated packet-loss), reconstruct the missing segment.

## 1 · Imports & Hyper-parameters

In [ ]:
import pathlib, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import librosa
from tqdm.auto import tqdm

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hyper-parameters ───────────────────────────────────────────────────────────
SAMPLE_RATE    = 22_050          # Hz (matches pre-processing)
CLIP_SAMPLES   = 22_050          # 1-second context window fed to the network
GAP_MIN_MS     = 80              # shortest packet-loss gap (ms)
GAP_MAX_MS     = 400             # longest  packet-loss gap (ms)

BATCH_SIZE     = 16
NUM_EPOCHS     = 50
LR_G           = 2e-4            # generator  learning rate
LR_D           = 1e-4            # discriminator learning rate
LAMBDA_L1      = 100.0           # weight for L1 reconstruction loss vs. adversarial

BASE_CHANNELS  = 32              # width of the first U-Net encoder block
NUM_ENC_BLOCKS = 5               # depth of the U-Net

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2 · Dataset — Loading, Segmentation & Masking

In [ ]:
class LJSpeechGapDataset(Dataset):
    """
    Streams fixed-length clips from LJ-Speech WAV files.
    For each clip it returns:
        masked  : waveform with a random zeroed-out gap  (model input)
        clean   : original waveform                      (ground truth)
        gap_mask: binary tensor — 1 where the gap is     (for loss masking)

    Directory layout expected:
        <root>/LJSpeech-1.1/wavs/*.wav          (already standardised to 22 050 Hz)
    """

    def __init__(
        self,
        root: str | pathlib.Path,
        clip_samples: int = CLIP_SAMPLES,
        sample_rate: int  = SAMPLE_RATE,
        gap_min_ms: int   = GAP_MIN_MS,
        gap_max_ms: int   = GAP_MAX_MS,
    ):
        self.root         = pathlib.Path(root)
        self.clip_samples = clip_samples
        self.sr           = sample_rate
        self.gap_min      = int(gap_min_ms  * sample_rate / 1000)
        self.gap_max      = int(gap_max_ms  * sample_rate / 1000)

        wav_dir = self.root / "LJSpeech-1.1" / "wavs"
        self.files = sorted(wav_dir.glob("*.wav"))
        if not self.files:
            raise FileNotFoundError(f"No .wav files found in {wav_dir}")

        # Pre-compute how many non-overlapping clips each file yields
        self._index: list[tuple[pathlib.Path, int]] = []
        for fpath in self.files:
            duration = librosa.get_duration(path=str(fpath))
            n_clips  = int(duration * self.sr) // self.clip_samples
            for i in range(n_clips):
                self._index.append((fpath, i))

    def __len__(self) -> int:
        return len(self._index)

    def __getitem__(self, idx: int):
        fpath, clip_idx = self._index[idx]

        # Load only the required segment (offset avoids loading the whole file)
        offset = clip_idx * self.clip_samples
        audio, _ = librosa.load(
            str(fpath),
            sr=self.sr,
            mono=True,
            offset=offset / self.sr,
            duration=self.clip_samples / self.sr,
        )

        # Pad or trim to exactly clip_samples (safety net)
        if len(audio) < self.clip_samples:
            audio = np.pad(audio, (0, self.clip_samples - len(audio)))
        else:
            audio = audio[: self.clip_samples]

        clean = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)  # [1, T]

        # Simulate packet loss: zero out a random contiguous gap
        gap_len   = random.randint(self.gap_min, self.gap_max)
        gap_start = random.randint(0, self.clip_samples - gap_len)

        masked    = clean.clone()
        masked[0, gap_start : gap_start + gap_len] = 0.0

        gap_mask  = torch.zeros(1, self.clip_samples, dtype=torch.float32)
        gap_mask[0, gap_start : gap_start + gap_len] = 1.0

        return masked, clean, gap_mask


# ── Quick sanity check ─────────────────────────────────────────────────────────
DATA_ROOT = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".")
# Adjust DATA_ROOT if your LJSpeech folder sits elsewhere:
# DATA_ROOT = pathlib.Path("/path/to/your/data")

dataset    = LJSpeechGapDataset(root=DATA_ROOT)
print(f"Total clips in dataset: {len(dataset)}")

val_size   = max(1, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=False)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## 3 · Model Architecture

### 3.1 — Generator: 1-D Convolutional U-Net
Encoder downsamples with `Conv1d`, decoder upsamples with `ConvTranspose1d`.  
Skip connections bridge matching encoder–decoder levels.

In [ ]:
class EncoderBlock(nn.Module):
    """Conv1d → BatchNorm → LeakyReLU  (stride 2 halves the time dimension)."""
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel,
                      stride=2, padding=kernel // 2, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    """ConvTranspose1d → BatchNorm → ReLU  (stride 2 doubles the time dimension).
    Accepts skip connection: input channels = 2 × out_ch (after cat)."""
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 15):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(in_ch, out_ch, kernel_size=kernel,
                               stride=2, padding=kernel // 2,
                               output_padding=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.block(x)
        # Align length in case of off-by-one from ConvTranspose
        if x.shape[-1] != skip.shape[-1]:
            x = x[..., : skip.shape[-1]]
        return torch.cat([x, skip], dim=1)   # channel-wise concatenation


class UNetGenerator(nn.Module):
    """
    1-D Convolutional U-Net.

    Input  : [B, 1, T]  — masked waveform (zeroed gap)
    Output : [B, 1, T]  — inpainted waveform (tanh bounded to [-1, 1])
    """
    def __init__(self, base_ch: int = BASE_CHANNELS, n_blocks: int = NUM_ENC_BLOCKS):
        super().__init__()

        # ── Encoder ───────────────────────────────────────────────────────────
        self.encoders = nn.ModuleList()
        in_ch = 1
        enc_channels = []
        for i in range(n_blocks):
            out_ch = base_ch * (2 ** i)
            self.encoders.append(EncoderBlock(in_ch, out_ch))
            enc_channels.append(out_ch)
            in_ch = out_ch

        # ── Bottleneck ────────────────────────────────────────────────────────
        bottleneck_ch = base_ch * (2 ** n_blocks)
        self.bottleneck = nn.Sequential(
            nn.Conv1d(in_ch, bottleneck_ch, kernel_size=15,
                      stride=2, padding=7, bias=False),
            nn.BatchNorm1d(bottleneck_ch),
            nn.ReLU(inplace=True),
        )

        # ── Decoder ───────────────────────────────────────────────────────────
        self.decoders = nn.ModuleList()
        in_ch = bottleneck_ch
        for i in reversed(range(n_blocks)):
            skip_ch = enc_channels[i]
            out_ch  = skip_ch          # mirror the encoder
            self.decoders.append(DecoderBlock(in_ch, out_ch))
            in_ch = out_ch + skip_ch   # after skip-cat: 2 × out_ch

        # ── Final projection ──────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.ConvTranspose1d(in_ch, 1, kernel_size=15,
                               stride=2, padding=7, output_padding=1, bias=True),
            nn.Tanh(),
        )

    def forward(self, x):
        skips = []
        h = x
        for enc in self.encoders:
            h = enc(h)
            skips.append(h)

        h = self.bottleneck(h)

        for dec, skip in zip(self.decoders, reversed(skips)):
            h = dec(h, skip)

        # Align final output to input length
        out = self.head(h)
        if out.shape[-1] != x.shape[-1]:
            out = out[..., : x.shape[-1]]
        return out


gen = UNetGenerator().to(DEVICE)
print(f"Generator parameters: {sum(p.numel() for p in gen.parameters()):,}")

### 3.2 — Discriminator: 1-D Conv Classifier
Receives a full waveform (real ground-truth **or** generated) and outputs a real/fake probability via Sigmoid.

In [ ]:
class Discriminator(nn.Module):
    """Write code inside this file 
    1-D Conv classifier.

    Input  : [B, 1, T]  — either ground-truth or generated waveform
    Output : [B, 1]     — real/fake score (sigmoid probability)
    """
    def __init__(self):
        super().__init__()
        def _block(in_ch, out_ch, stride=2):
            return nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=15,
                          stride=stride, padding=7, bias=False),
                nn.BatchNorm1d(out_ch),
                nn.LeakyReLU(0.2, inplace=True),
            )

        self.net = nn.Sequential(
            _block(1,   32,  stride=2),
            _block(32,  64,  stride=2),
            _block(64,  128, stride=2),
            _block(128, 256, stride=2),
            _block(256, 512, stride=2),
            nn.AdaptiveAvgPool1d(1),        # global average pool → [B, 512, 1]
        )
        self.head = nn.Sequential(
            nn.Flatten(),                   # [B, 512]
            nn.Linear(512, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.head(self.net(x))


disc = Discriminator().to(DEVICE)
print(f"Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}")

### 4 · Loss Functions & Optimisers

- **Adversarial loss** — Binary Cross-Entropy on real/fake discriminator scores  
- **L1 Reconstruction loss** — MAE between the generated and ground-truth waveform (weighted by `LAMBDA_L1`)  
- Total generator loss = `L_adv + λ · L1`

In [ ]:
adversarial_loss = nn.BCELoss()
reconstruction_loss = nn.L1Loss()

opt_G = optim.Adam(gen.parameters(),  lr=LR_G, betas=(0.5, 0.999))
opt_D = optim.Adam(disc.parameters(), lr=LR_D, betas=(0.5, 0.999))

# Linear learning-rate decay over the last 25 % of training
def lr_lambda(epoch):
    decay_start = int(NUM_EPOCHS * 0.75)
    if epoch < decay_start:
        return 1.0
    return 1.0 - (epoch - decay_start) / max(1, NUM_EPOCHS - decay_start)

scheduler_G = optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
scheduler_D = optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)

print("Optimisers and schedulers ready.")

### 5 · Training Loop

In [ ]:
history = {"epoch": [], "loss_D": [], "loss_G": [], "loss_L1": [], "val_loss_L1": []}

for epoch in range(1, NUM_EPOCHS + 1):
    gen.train()
    disc.train()

    running_D, running_G, running_L1 = 0.0, 0.0, 0.0
    t0 = time.time()

    for masked, clean, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False):
        masked = masked.to(DEVICE)
        clean  = clean.to(DEVICE)
        B      = clean.size(0)

        real_labels = torch.ones (B, 1, device=DEVICE)
        fake_labels = torch.zeros(B, 1, device=DEVICE)

        # ── (1) Train Discriminator ────────────────────────────────────────────
        opt_D.zero_grad()

        fake_audio = gen(masked).detach()          # stop gradients to generator

        loss_D_real = adversarial_loss(disc(clean),      real_labels)
        loss_D_fake = adversarial_loss(disc(fake_audio), fake_labels)
        loss_D = 0.5 * (loss_D_real + loss_D_fake)

        loss_D.backward()
        opt_D.step()

        # ── (2) Train Generator ────────────────────────────────────────────────
        opt_G.zero_grad()

        fake_audio  = gen(masked)
        loss_G_adv  = adversarial_loss(disc(fake_audio), real_labels)   # fool disc
        loss_G_l1   = reconstruction_loss(fake_audio, clean) * LAMBDA_L1
        loss_G      = loss_G_adv + loss_G_l1

        loss_G.backward()
        opt_G.step()

        running_D  += loss_D.item()
        running_G  += loss_G_adv.item()
        running_L1 += loss_G_l1.item() / LAMBDA_L1      # store raw L1

    scheduler_G.step()
    scheduler_D.step()

    # ── Validation ────────────────────────────────────────────────────────────
    gen.eval()
    val_l1 = 0.0
    with torch.no_grad():
        for masked_v, clean_v, _ in val_loader:
            masked_v = masked_v.to(DEVICE)
            clean_v  = clean_v.to(DEVICE)
            fake_v   = gen(masked_v)
            val_l1  += reconstruction_loss(fake_v, clean_v).item()
    val_l1 /= len(val_loader)

    n = len(train_loader)
    history["epoch"].append(epoch)
    history["loss_D"].append(running_D / n)
    history["loss_G"].append(running_G / n)
    history["loss_L1"].append(running_L1 / n)
    history["val_loss_L1"].append(val_l1)

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:>3}/{NUM_EPOCHS}  "
        f"D: {running_D/n:.4f}  "
        f"G_adv: {running_G/n:.4f}  "
        f"L1: {running_L1/n:.4f}  "
        f"Val_L1: {val_l1:.4f}  "
        f"({elapsed:.1f}s)"
    )

### 6 · Save the Trained Models

Both the generator and discriminator checkpoints are stored in the same `model/` directory as this notebook.

In [ ]:
SAVE_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".")

gen_path  = SAVE_DIR / "sandhi_generator.pth"
disc_path = SAVE_DIR / "sandhi_discriminator.pth"

torch.save(
    {
        "epoch":        NUM_EPOCHS,
        "model_state":  gen.state_dict(),
        "optim_state":  opt_G.state_dict(),
        "history":      history,
        "hparams": {
            "sample_rate":   SAMPLE_RATE,
            "clip_samples":  CLIP_SAMPLES,
            "gap_min_ms":    GAP_MIN_MS,
            "gap_max_ms":    GAP_MAX_MS,
            "base_channels": BASE_CHANNELS,
            "num_enc_blocks": NUM_ENC_BLOCKS,
        },
    },
    gen_path,
)

torch.save(
    {
        "epoch":       NUM_EPOCHS,
        "model_state": disc.state_dict(),
        "optim_state": opt_D.state_dict(),
    },
    disc_path,
)

print(f"Generator  saved → {gen_path}")
print(f"Discriminator saved → {disc_path}")

### 7 · Training Curves

In [ ]:
try:
    import matplotlib.pyplot as plt

    epochs = history["epoch"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history["loss_D"], label="Discriminator Loss")
    axes[0].set_title("Discriminator Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
    axes[0].legend()

    axes[1].plot(epochs, history["loss_G"], label="Generator Adv Loss")
    axes[1].set_title("Generator Adversarial Loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("BCE Loss")
    axes[1].legend()

    axes[2].plot(epochs, history["loss_L1"],     label="Train L1")
    axes[2].plot(epochs, history["val_loss_L1"], label="Val L1", linestyle="--")
    axes[2].set_title("L1 Reconstruction Loss")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("L1 Loss")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(SAVE_DIR / "training_curves.png", dpi=150)
    plt.show()
    print("Training curves saved.")
except ImportError:
    print("matplotlib not installed — skipping plot.")